## Data Cleaning: Creating `sources` and `emissions_records` tables
The purpose of this script is to create two cleaned tables from the electricity-generated emissions and the emissions from non-residential onsite building usage based on data compiled from Climate TRACE. Two tables are created: `sources` (containing the source name and other attributes) and `emissions_records` (containing the emissions quantity for source records + other attributes). 

### Import Packages 

In [95]:
import pandas as pd

In [96]:
power = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/power/electricity-generation_emissions_sources_v5_5_0.csv')
non_res = pd.read_csv('/Users/vedikashirtekar/MEDS/EDS-213/eds-213-labs/data/DATA/buildings/non-residential-onsite-fuel-usage_emissions_sources_v5_5_0.csv')


It's always a good idea to explore the datatypes of each of the data frames.

In [97]:
# Explore the data structures
# print(non_res.shape, power.shape)
print(non_res.dtypes)

source_id                   int64
source_name                object
source_type               float64
iso3_country               object
sector                     object
subsector                  object
start_time                 object
end_time                   object
lat                       float64
lon                       float64
geometry_ref               object
gas                        object
emissions_quantity        float64
temporal_granularity       object
activity                  float64
activity_units             object
emissions_factor          float64
emissions_factor_units     object
capacity                  float64
capacity_units             object
capacity_factor           float64
other1                     object
other1_def                 object
other2                     object
other2_def                 object
other3                     object
other3_def                 object
other4                    float64
other4_def                 object
other5        

In [98]:
print(power.dtypes)

source_id                   int64
source_name                object
source_type                object
iso3_country               object
sector                     object
subsector                  object
start_time                 object
end_time                   object
lat                       float64
lon                       float64
geometry_ref              float64
gas                        object
emissions_quantity        float64
temporal_granularity       object
activity                  float64
activity_units             object
emissions_factor          float64
emissions_factor_units     object
capacity                  float64
capacity_units             object
capacity_factor           float64
other1                     object
other1_def                 object
other2                    float64
other2_def                 object
other3                    float64
other3_def                 object
other4                    float64
other4_def                 object
other5        

In [ ]:
# Drop always null or useless columns
# Some columns (source_type, geometry_ref, sector, and subsector) is entirely null, not relevant, are has a constant value
drop_cols = ['source_type', 'geometry_ref', 'sector', 'subsector', 'created_date', 'modified_date']
non_res = non_res.drop(columns=drop_cols, errors='ignore')
power = power.drop(columns=drop_cols, errors='ignore')

In [79]:
# Parse datetime columns
non_res['start_time'] = pd.to_datetime(non_res['start_time'])
non_res['end_time'] = pd.to_datetime(non_res['end_time'])
# same for power
power['start_time'] = pd.to_datetime(power['start_time'])
power['end_time'] = pd.to_datetime(power['end_time'])

In [80]:
# Are there any duplicates? 
# Valid dataset should have one row per source_id + start_time + gas combination
dupes = non_res.duplicated(subset=['source_id', 'start_time', 'gas'])
print(dupes.sum())

0


In [81]:
# Check for nulls and outliers in numeric fields
numeric_cols = ['emissions_quantity', 'activity', 'emissions_factor', 'capacity', 'lat', 'lon']
print(non_res[numeric_cols].describe())
print(non_res[numeric_cols].isnull().sum())

       emissions_quantity      activity  emissions_factor      capacity  \
count       207888.000000  2.078880e+05      2.078880e+05  2.078880e+05   
mean          5293.175706  8.615382e+07      7.349123e-05  2.670397e+06   
std          24256.979637  3.883335e+08      8.532392e-05  1.141074e+07   
min              0.000000  0.000000e+00      9.587420e-07  0.000000e+00   
25%            152.193098  2.354243e+06      5.001449e-05  7.977713e+04   
50%            593.892610  9.499643e+06      6.296034e-05  3.128240e+05   
75%           2641.845250  4.269371e+07      8.130056e-05  1.381689e+06   
max         981715.594000  1.618478e+10      2.437498e-03  3.365205e+08   

                 lat            lon  
count  192028.000000  192028.000000  
mean       38.454181     -92.203442  
std         5.276979      12.787112  
min        19.596185    -164.442490  
25%        34.698653     -98.207827  
50%        38.399181     -90.357245  
75%        41.851047     -83.431854  
max        69.356849

In [82]:
# Same for power
#numeric_cols = ['emissions_quantity', 'activity', 'emissions_factor', 'capacity', 'lat', 'lon']
print(power[numeric_cols].describe())
print(power[numeric_cols].isnull().sum())

       emissions_quantity      activity  emissions_factor       capacity  \
count        1.548790e+05  1.548790e+05     154879.000000  154879.000000   
mean         4.834783e+04  9.023929e+04          0.426030     318.851836   
std          1.256916e+05  1.852249e+05          0.310532     519.207310   
min          0.000000e+00  0.000000e+00          0.000000       0.000000   
25%          4.760500e+02  1.803000e+03          0.320950      15.900000   
50%          4.402000e+03  1.222000e+04          0.449360      70.000000   
75%          3.605000e+04  7.439500e+04          0.513628     432.000000   
max          1.958000e+06  2.300000e+06          1.473934    4329.600000   

                 lat            lon  
count  154879.000000  154879.000000  
mean       38.097927     -93.079859  
std         5.928407      17.128114  
min        19.631600    -166.553200  
25%        33.869700     -98.322800  
50%        38.985800     -89.589100  
75%        41.663100     -81.059200  
max        

In [83]:
# Drop "other" columns in non_res and power
other_cols = [c for c in non_res.columns if c.startswith('other')]
non_res = non_res.drop(columns=other_cols)
#power = power.drop(columns=other_cols)

In [84]:
# Drop "other" columns in power
other_cols = [c for c in power.columns if c.startswith('other')]
power = power.drop(columns=other_cols)

In [85]:
# Create sources table
sources = non_res[['source_id', 'source_name', 'iso3_country', 'lat', 'lon', 
                    'capacity', 'capacity_units', 'capacity_factor']].drop_duplicates(subset='source_id')
sources_power = power[['source_id', 'source_name', 'iso3_country', 'lat', 'lon',
                        'capacity', 'capacity_units', 'capacity_factor']].drop_duplicates(subset='source_id')

sources = pd.concat([sources, sources_power]).drop_duplicates(subset='source_id').reset_index(drop=True)

In [86]:
record_cols = ['source_id', 'start_time', 'end_time', 'gas', 'emissions_quantity',
               'temporal_granularity', 'activity', 'activity_units', 
               'emissions_factor', 'emissions_factor_units']

non_records = non_res[record_cols].copy()
non_records['sector'] = 'buildings'

power_records = power[record_cols].copy()
power_records['sector'] = 'power'

emission_records = pd.concat([non_records, power_records]).reset_index(drop=True)

In [88]:
# Export
sources.to_csv('data/new_tables/sources.csv', index=False)
emission_records.to_csv('data/new_tables/emission_records.csv', index=False)

In [44]:
emission_records.dtypes

source_id                          int64
start_time                datetime64[ns]
end_time                  datetime64[ns]
gas                               object
emissions_quantity               float64
temporal_granularity              object
activity                         float64
activity_units                    object
emissions_factor                 float64
emissions_factor_units            object
sector                            object
dtype: object

In [50]:
sources

,source_id,source_name,iso3_country,lat,lon,capacity,capacity_units,capacity_factor
0,1054027,Abbeville County,USA,34.223165,-82.458302,201605.089,m^2,54.991962
1,34157084,Abilene Urban Area,USA,NaN,NaN,2922695.433,m^2,57.514333
2,1052820,Acadia Parish,USA,30.291811,-92.411828,512078.827,m^2,53.181906
3,1054530,Accomack County,USA,37.752250,-75.628073,211725.066,m^2,53.388962
4,1052256,Ada County,USA,43.450432,-116.240313,8879498.010,m^2,45.848097
...,...,...,...,...,...,...,...,...
5942,25449346,Zeeland Generating Station,USA,42.820600,-85.997500,968.200,MW,328.134683
5943,25456804,Zeltmann,USA,40.788900,-73.906900,528.000,MW,398.484848
5944,25456644,Zion Energy Center,USA,42.477600,-87.895000,596.700,MW,34.774594
5945,25449594,Zorn,USA,38.280300,-85.702300,18.000,MW,261.333333


In [73]:
emission_records

,source_id,start_time,end_time,gas,emissions_quantity,temporal_granularity,activity,activity_units,emissions_factor,emissions_factor_units,sector
0,1054027,2021-01-01,2021-01-31,co2,740.024,month,1.108666e+07,MJ,0.000067,t of CO2 per MJ,buildings
1,1054027,2021-02-01,2021-02-28,co2,652.852,month,9.777332e+06,MJ,0.000067,t of CO2 per MJ,buildings
2,1054027,2021-03-01,2021-03-31,co2,475.286,month,7.118559e+06,MJ,0.000067,t of CO2 per MJ,buildings
3,1054027,2021-04-01,2021-04-30,co2,394.517,month,5.905576e+06,MJ,0.000067,t of CO2 per MJ,buildings
4,1054027,2021-05-01,2021-05-31,co2,317.548,month,4.752445e+06,MJ,0.000067,t of CO2 per MJ,buildings
...,...,...,...,...,...,...,...,...,...,...,...
362762,41627864,2025-09-01,2025-09-30,co2,0.000,month,0.000000e+00,MWh,0.000000,t of CO2 per MWh,power
362763,41627864,2025-10-01,2025-10-31,co2,0.000,month,0.000000e+00,MWh,0.000000,t of CO2 per MWh,power
362764,41627864,2025-11-01,2025-11-30,co2,0.000,month,0.000000e+00,MWh,0.000000,t of CO2 per MWh,power
362765,41627864,2025-12-01,2025-12-31,co2,0.000,month,0.000000e+00,MWh,0.000000,t of CO2 per MWh,power


In [94]:
emission_records["subsector"].unique()

KeyError: 'subsector'